# Ceasul Cuantic al Iașului
## Nivelul 1 — Inima ceasului

*Qiskit Fall Fest Iași 2026 · Facultatea de Automatică și Calculatoare, TUIASI*

> ### Palatul Culturii, ora 08:00
>
> Iași, toamna anului 2036. În turnul Palatului Culturii, lângă cele opt clopote care cântă *Hora Unirii* la fiecare oră, bate inima rețelei cuantice a orașului: procesorul **QIași-1**.
>
> În noaptea dinaintea Qiskit Fall Fest, clopotele au tăcut. Ceasul s-a oprit la **03:14**. Un intrus care semnează doar **„Umbra”** a dereglat qubiții, a rupt legăturile dintre noduri și a ascuns cheia de repornire într-unul din opt locuri ale orașului.
>
> **Voi sunteți Echipa de Intervenție.** Urcați scările înguste ale turnului. Pe ecranul de control clipește un mesaj: *QUBIT DECALIBRAT*. Ca să pornească din nou, inima ceasului trebuie să „bată” cu probabilități exacte — iar Umbra le-a dat peste cap.
>
> Prima misiune: învățați cum se comportă un qubit și calibrați-l.

### Cum se joacă

- Provocarea are **6 niveluri** și **8 clopote**. Un clopot se aprinde când rezolvați o etapă importantă din poveste.
- Fiecare sarcină verificată corect aduce **puncte**. Celulele `verifica_...()` vă spun imediat dacă ați reușit.
- Verificarea se face pe **serverul organizatorilor**, deci aveți nevoie de rețea. Clopotele voastre apar imediat și pe **ecranul mare** din sală.
- Blocați? Vorbiți întâi cu mentorul (gratuit). Apoi puteți cere `indiciu(nivel, 1)` (costă 10% din punctele nivelului) sau `indiciu(nivel, 2)` (costă 20%).
- Casetele galbene **Provocare extra** sunt opționale și aduc puncte în plus.
- În celulele de cod, căutați `# COMPLETEAZĂ AICI`.

### Ce veți învăța în acest nivel

- ce este un **qubit** și cum diferă de un bit;
- ce înseamnă **superpoziție** și **măsurare**;
- cum **rotiți** un qubit ca să obțineți exact probabilitatea dorită.

## 0. Pregătire

Rulați celula de mai jos (Shift + Enter). Scrieți numele echipei voastre între ghilimele.

In [2]:
# Rulați această celulă prima. Ea încarcă Qiskit și uneltele provocării.
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from ceas_cuantic import *

ECHIPA = "Don't quit, qubit!"   # COMPLETEAZĂ AICI — același nume în toate notebook-urile
COD_ECHIPA = "UP2U-JRC"     # COMPLETEAZĂ AICI — codul primit de la organizatori
start(ECHIPA, COD_ECHIPA)

ModuleNotFoundError: No module named 'qiskit'

## 1. Qubitul

Un **bit** clasic este fie 0, fie 1. Un **qubit** poate fi într-o combinație a celor două:

$$|\psi\rangle = \alpha\,|0\rangle + \beta\,|1\rangle$$

Numerele $\alpha$ și $\beta$ se numesc **amplitudini** (pot fi chiar numere complexe — le știți din clasa a X-a). Când **măsurăm** qubitul, obținem:

- **0** cu probabilitatea $|\alpha|^2$,
- **1** cu probabilitatea $|\beta|^2$,

iar $|\alpha|^2 + |\beta|^2 = 1$. După măsurare, qubitul „alege” și rămâne în starea măsurată.

Gândiți-vă la o monedă care se învârte pe masă: cât timp se învârte nu e nici cap, nici pajură. Când o opriți cu palma, devine una dintre ele.

Orice qubit pornește în starea $|0\rangle$. Două porți importante:

| Poartă | Ce face | Pe scurt |
| --- | ------------- | ----- |
| **X** | $\lvert 0\rangle \to \lvert 1\rangle$ $\lvert 1\rangle \to \lvert 0\rangle$ | NOT-ul cuantic |
| **H** (Hadamard) | $\lvert 0\rangle \to \frac{1}{\sqrt2}(\lvert 0\rangle + \lvert 1\rangle) = \lvert +\rangle$ | creează superpoziția 50/50 |

Iată un circuit cu un qubit și o poartă H. `Statevector` ne arată amplitudinile (lucru posibil doar în simulare!), iar `masoara` rulează circuitul de 1000 de ori, ca un experiment real.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
display(qc.draw("mpl"))

stare = Statevector(qc)
print("Amplitudini:", np.round(stare.data, 3))
print("Probabilități:", stare.probabilities_dict())

counts = masoara(qc, shots=1000)
print("Rezultatele a 1000 de măsurători:", counts)
plot_histogram(counts)

<div class="alert alert-block alert-success">
<b>Sarcina 1A — Primii qubiți (40 de puncte)</b>

Construiți trei circuite cu câte un qubit:
<ul>
<li><code>qc_zero</code> — qubitul rămâne în starea |0⟩;</li>
<li><code>qc_unu</code> — qubitul ajunge în starea |1⟩;</li>
<li><code>qc_plus</code> — qubitul ajunge în starea |+⟩ (50% 0, 50% 1).</li>
</ul>
Nu adăugați măsurători: verificatorul se uită direct la starea qubitului.
</div>

In [ ]:
qc_zero = QuantumCircuit(1)
# COMPLETEAZĂ AICI (dacă e nevoie)

qc_unu = QuantumCircuit(1)
# COMPLETEAZĂ AICI

qc_plus = QuantumCircuit(1)
# COMPLETEAZĂ AICI

plot_histogram([masoara(qc_zero), masoara(qc_unu), masoara(qc_plus)],
               legend=["qc_zero", "qc_unu", "qc_plus"])

**Discutați în echipă:** `qc_plus` are probabilitatea exact 50% pentru 1. De ce nu obțineți exact 500 din 1000? Ce s-ar întâmpla cu 100.000 de măsurători?

In [ ]:
verifica_1A(qc_zero, qc_unu, qc_plus)

## 2. Rotații: cum alegem orice probabilitate

H dă mereu 50/50. Pentru inima ceasului avem nevoie de probabilități exacte, de exemplu 25%. Folosim poarta de **rotație** $R_y(\theta)$, care rotește qubitul cu unghiul $\theta$:

$$R_y(\theta) = \begin{pmatrix} \cos\frac{\theta}{2} & -\sin\frac{\theta}{2} \\ \sin\frac{\theta}{2} & \cos\frac{\theta}{2} \end{pmatrix}, \qquad R_y(\theta)\,|0\rangle = \cos\tfrac{\theta}{2}\,|0\rangle + \sin\tfrac{\theta}{2}\,|1\rangle$$

Deci probabilitatea de a măsura 1 este

$$P(1) = \sin^2\frac{\theta}{2}.$$

| $\theta$ | 0 | $\frac{\pi}{2}$ | $\pi$ |
| --- | --- | --- | --- |
| $P(1)$ | 0 | 0,5 | 1 |

Rulați celula de mai jos și mutați unghiul ca să vedeți cum se schimbă probabilitatea.

In [ ]:
theta = np.pi / 2          # încercați și alte valori: 0, np.pi / 4, np.pi, ...
qc = QuantumCircuit(1)
qc.ry(theta, 0)
print(f"θ = {theta:.3f} rad   →   P(1) = {Statevector(qc).probabilities()[1]:.3f}")

<div class="alert alert-block alert-success">
<b>Sarcina 1B — Calibrarea inimii ceasului (60 de puncte · clopotul 1)</b>

Inima ceasului are nevoie de qubiți care dau „1” cu o probabilitate exactă <code>p</code> (de exemplu 25%, 50%, 90%).<br>
Completați funcția <code>calibreaza(p)</code>: ea trebuie să returneze un circuit cu un qubit pentru care P(1) = p.<br><br>
Pe hârtie întâi: pentru p = 0,25, ce unghi θ obțineți din P(1) = sin²(θ/2)?<br>
Indiciu de matematică: funcția inversă a sinusului este <code>np.arcsin</code>, iar radicalul este <code>np.sqrt</code>.
</div>

In [ ]:
def calibreaza(p):
    """Returnează un circuit cu 1 qubit pentru care P(1) = p."""
    qc = QuantumCircuit(1)
    theta = 0   # COMPLETEAZĂ AICI: calculați unghiul din p
    qc.ry(theta, 0)
    return qc


for p in [0.25, 0.5, 0.9]:
    counts = masoara(calibreaza(p), shots=2000)
    print(f"țintă P(1) = {p:.2f}   măsurat: {counts.get('1', 0) / 2000:.3f}")

In [ ]:
verifica_1B(calibreaza)

<div class="alert alert-block alert-warning">
<b>Provocare extra — Rotația defectă (+50 de puncte)</b>

Umbra a stricat și motorul de rotație: funcția <code>ry_defect(qc, theta)</code> rotește qubitul cu un unghi de <b>k</b> ori mai mare decât cel cerut. Factorul k este secret.<br><br>
Așa se calibrează și qubiții reali (experimentul se numește <i>Rabi</i>): variați unghiul, măsurați P(1), desenați graficul și găsiți k.<br>
Pentru rotația defectă, P(1) = sin²(kθ/2). Graficul revine la 0 după o perioadă de 2π/k, iar primul maxim este la θ = π/k.<br>
Precizie cerută: ±0,05.

Hint: rulati prima data celula pentru a vi se afisa graficul. Ulterior completati sarcina ceruta
</div>

In [ ]:
unghiuri = np.linspace(0, 4 * np.pi, 101)
circuite = []
for theta in unghiuri:
    qc = QuantumCircuit(1)
    ry_defect(qc, theta)
    circuite.append(qc)
rezultate = masoara_toate(circuite, shots=500)      # toate circuitele odată, pe server
p1 = [c.get("1", 0) / 500 for c in rezultate]

plt.plot(unghiuri, p1, "o-", markersize=3)
plt.xlabel("unghiul cerut θ (radiani)")
plt.ylabel("P(1) măsurat")
plt.grid(alpha=0.3)
plt.show()

k_estimat = 1.0   # INLOCUIESTE k AICI: citiți k din grafic (sau potriviți o curbă)
verifica_extra1(k_estimat)

<div class="alert alert-block alert-info">
<b>La AC studiezi asta</b>

Ce ați făcut în extra este o <b>buclă de reglare</b>: măsori, compari cu ținta, corectezi. Calculatoarele cuantice IBM își recalibrează qubiții regulat exact așa.<br>
La Automatică studiezi cum se proiectează astfel de bucle pentru orice sistem: un motor, o dronă, o rețea electrică — sau un qubit.
</div>

> ### Inima bate din nou
>
> Pe ecranul turnului apare: *QUBIT CALIBRAT*. Primul clopot se leagănă ușor. Dar rețeaua tot nu răspunde: semnalele se pierd undeva sub Biblioteca Centrală Universitară...

## Sfârșitul nivelului 1

Rulați celula de mai jos ca să vedeți turnul echipei.

Când sunteți gata, deschideți **Nivelul_2_Piesa_sabotata.ipynb**.

In [ ]:
turn()